# DeepSORT Overview

# DeepSORT Video Summary

**Goal:** Assign IDs to cars and keep the same ID for the same car throughout the video.

---

## DeepSORT Algorithm Goal

* Detect objects and track them over time.
* Determine where objects are going (such as the case of traffic tracking).

---

## SORT – Simple Online Realtime Tracking

### SimpleSORT

Uses bounding box prediction, Kalman filter, and IoU matching techniques.

**Steps:**

1. **Bounding Box Detection**

   * Detect the location of objects.
   * Can use any CNN, YOLO, or RCNN.
   * For my implementation: We will YOLOv11 as Step 1.
   * Once bounding boxes are generated, move to Step 2.

2. **Kalman Filter**

   * Simple linear approximation to predict future object locations.
   * Helps detect if the object we tracked is the same as the one currently observed.
   * Handles occlusion.
   * Assumes a linear velocity model and generates a new set of predictions.
   * Outputs a probability distribution of where the object could be.

3. **IoU Matching (Intersection over Union)**

   * Quantitatively compares two bounding boxes.
   * For `n` objects and `n` IDs, find matches that maximize IoU scores.
   * SORT uses the Hungarian algorithm to solve this linear assignment problem efficiently (`O(n^2)`).

---

## DeepSORT – An Extension of SimpleSORT

* **Uses Deep Appearance Descriptor**:

  * CNN trained to recognize **similar objects** across different images (e.g., the same person in different frames).
  * **Input:** Cropped image from the bounding box.
  * **Output:** Encoded vectors (cosine distance) for object comparison:

    * Cosine of 0° → 1 → very similar
    * Cosine of 90° → 0 → dissimilar

* Combines CNN output with Kalman Filter output using **Mahalanobis distance**:

  * Similar objects → similar distances to origin.
  * Dissimilar objects → dissimilar distances.

---

### Mahalanobis Distance

* Measures distance accounting for data spread in multiple dimensions.
* Normalize and transform coordinates, then calculate distance to origin.
* Used alongside cosine distance for the Hungarian algorithm in DeepSORT.

---

### Cascade Matching

* Takes the **temporal dimension** into account.
* Considers recent detections/IDs and older detections/IDs for matching.



![Architecture](DeepSORT_excalidraw.png)


# Summary of Key Components


# DeepSORT Architecture – Key Components

DeepSORT extends SORT by combining motion prediction (Kalman filter) and appearance information (deep features) for robust object tracking.

---

## 1. Bounding Box Detection
- Detect objects in each frame using a CNN (e.g., YOLOv11).  
- **Output:** bounding box coordinates `[x, y, w, h]`.  

---

## 2. Kalman Filter for Motion Prediction

The Kalman filter predicts where each tracked object is likely to move and updates its estimate when new measurements arrive.

### State Representation
The object state is represented as:

$$
\mathbf{x} = [x, y, a, h, \dot{x}, \dot{y}, \dot{a}, \dot{h}]^T
$$

Where:  
- \(x, y\) – center coordinates of the bounding box  
- \(a\) – aspect ratio  
- \(h\) – height  
- \(\dot{x}, \dot{y}, \dot{a}, \dot{h}\) – velocities

---

### Motion Model

We predict the next state using a linear motion model:

$$
\mathbf{x}_k = \mathbf{F} \mathbf{x}_{k-1} + \mathbf{w}_{k-1}
$$

- F is the motion matrix  
- w_k-1represents process noise  


```python
mean = np.dot(self._motion_mat, mean)
covariance = self._motion_mat @ covariance @ self._motion_mat.T + motion_cov
````

---

### Measurement Update

The measurement vector is:

$$
\mathbf{z}_k = [x, y, a, h]^T
$$

Project the predicted state to measurement space:

$$
\mathbf{z}_{pred} = \mathbf{H} \mathbf{x}_k
$$

Compute the innovation (difference between measurement and prediction):

$$
\mathbf{y} = \mathbf{z}*k - \mathbf{z}*{pred}
$$

Compute the Kalman gain:

$$
\mathbf{K} = \Sigma \mathbf{H}^T (\mathbf{H} \Sigma \mathbf{H}^T + R)^{-1}
$$

Update the state and covariance:

$$
\mathbf{x}_k = \mathbf{x}_k + \mathbf{K} \mathbf{y}
$$

$$
\Sigma_k = \Sigma_k - \mathbf{K} (\mathbf{H} \Sigma \mathbf{H}^T + R) \mathbf{K}^T
$$



```python
new_mean = mean + np.dot(innovation, kalman_gain.T)
new_covariance = covariance - kalman_gain @ projected_cov @ kalman_gain.T
```

---

### Gating / Mahalanobis Distance

Mahalanobis distance measures how likely a detection belongs to a track:

$$
d^2 = (\mathbf{z} - \hat{\mathbf{z}})^T \Sigma^{-1} (\mathbf{z} - \hat{\mathbf{z}})
$$

* Can be computed using only position or the full state
* Helps reject unlikely matches

---

## 3. IoU Matching (SORT)

* Compute Intersection-over-Union (IoU) between predicted and detected boxes
* Assign detections to tracks by maximizing IoU
* Solved using the **Hungarian Algorithm**

### Hungarian Algorithm

* Solves the linear assignment problem efficiently
* Input: cost matrix (negative IoU or distance between tracks and detections)
* Output: optimal one-to-one assignments that minimize total cost
* Implemented by using linear_sum_assignment from SciPy

---

## 4. Deep Appearance Descriptor

* A CNN generates a feature vector for each detected object (from the cropped image)
* Measure similarity using **cosine similarity**:

$$
\text{cosine similarity}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{|\mathbf{a}| , |\mathbf{b}|}
$$

* Combined with Mahalanobis distance to compute the final assignment cost

---

## 5. Cascade Matching

* Prioritizes matching **recent tracks** first
* Older tracks are considered only if recent tracks do not match
* Helps maintain track continuity under occlusion

```



In [ ]:
import cv2
import numpy as np
import time
from ultralytics import YOLO
import colorsys
import random
import os

import torch
import torchvision.transforms as transforms
from torchvision.models import resnet50, ResNet50_Weights

import scipy.linalg
from scipy.optimize import linear_sum_assignment

In [5]:
class Detection(object):
    """
    This class represents a bounding box detection in a single image.
    """

    def __init__(self, tlwh, confidence, class_name, feature):
        # self.tlwh = np.asarray(tlwh, dtype=np.float) Deprecated
        self.tlwh = np.asarray(tlwh, dtype=np.float64)
        self.confidence = float(confidence)
        self.class_name = class_name
        self.feature = np.asarray(feature, dtype=np.float32)

    def get_class(self):
        return self.class_name

    def to_tlbr(self):
        """Convert bounding box to format `(min x, min y, max x, max y)`"""
        ret = self.tlwh.copy()
        ret[2:] += ret[:2]
        return ret

    def to_xyah(self):
        """Convert bounding box to format `(center x, center y, aspect ratio, height)`"""
        ret = self.tlwh.copy()
        ret[:2] += ret[2:] / 2
        ret[2] /= ret[3]
        return ret

In [ ]:
"""
GPU-ENABLED PyTorch Feature Extractor for DeepSORT
"""


class SimpleFeatureExtractor:
    """GPU-accelerated feature extractor using PyTorch."""
    
    def __init__(self, device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.device = device
        print(f"🚀 Using device: {self.device}")
        
        # Load pretrained ResNet50
        self.model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        # Remove the final classification layer
        self.model = torch.nn.Sequential(*list(self.model.children())[:-1])
        self.model.eval()
        self.model.to(device)
        
        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((128, 64)),  # DeepSORT standard size
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
        
        self.feature_dim = 2048
        self.image_shape = (128, 64, 3)
    
    def __call__(self, image_patches, batch_size=32):
        """Extract features from image patches using GPU."""
        features = []
        
        with torch.no_grad():
            for i in range(0, len(image_patches), batch_size):
                batch = image_patches[i:i+batch_size]
                batch_tensors = torch.stack([self.transform(img) for img in batch])
                batch_tensors = batch_tensors.to(self.device)  # GPU!
                
                # Extract features on GPU
                batch_features = self.model(batch_tensors)
                batch_features = batch_features.squeeze().cpu().numpy()
                
                if len(batch_features.shape) == 1:
                    batch_features = batch_features.reshape(1, -1)
                
                features.append(batch_features)
        
        return np.vstack(features)


def extract_image_patch(image, bbox, patch_shape):
    """Extract image patch from bounding box."""
    bbox = np.array(bbox)
    if patch_shape is not None:
        target_aspect = float(patch_shape[1]) / patch_shape[0]
        new_width = target_aspect * bbox[3]
        bbox[0] -= (new_width - bbox[2]) / 2
        bbox[2] = new_width

    bbox[2:] += bbox[:2]
    bbox = bbox.astype(int)
    bbox[:2] = np.maximum(0, bbox[:2])
    bbox[2:] = np.minimum(np.asarray(image.shape[:2][::-1]) - 1, bbox[2:])
    if np.any(bbox[:2] >= bbox[2:]):
        return None
    sx, sy, ex, ey = bbox
    image = image[sy:ey, sx:ex]
    image = cv2.resize(image, tuple(patch_shape[::-1]))
    return image


def create_box_encoder(batch_size=32, device='cuda' if torch.cuda.is_available() else 'cpu'):
    """Create GPU-accelerated box encoder (auto-detects GPU)."""
    feature_extractor = SimpleFeatureExtractor(device=device)
    image_shape = feature_extractor.image_shape

    def encoder(image, boxes):
        image_patches = []
        for box in boxes:
            patch = extract_image_patch(image, box, image_shape[:2])
            if patch is None:
                print("WARNING: Failed to extract image patch: %s." % str(box))
                patch = np.random.uniform(0.0, 255.0, image_shape).astype(np.uint8)
            image_patches.append(patch)
        image_patches = np.asarray(image_patches)
        return feature_extractor(image_patches, batch_size)

    return encoder

In [7]:
chi2inv95 = {1: 3.8415, 2: 5.9915, 3: 7.8147, 4: 9.4877, 5: 11.070, 6: 12.592, 7: 14.067, 8: 15.507, 9: 16.919}


class KalmanFilter(object):
    """A simple Kalman filter for tracking bounding boxes in image space."""

    def __init__(self):
        ndim, dt = 4, 1.0
        self._motion_mat = np.eye(2 * ndim, 2 * ndim)
        for i in range(ndim):
            self._motion_mat[i, ndim + i] = dt
        self._update_mat = np.eye(ndim, 2 * ndim)
        self._std_weight_position = 1.0 / 20
        self._std_weight_velocity = 1.0 / 160

    def initiate(self, measurement):
        mean_pos = measurement
        mean_vel = np.zeros_like(mean_pos)
        mean = np.r_[mean_pos, mean_vel]
        std = [
            2 * self._std_weight_position * measurement[3],
            2 * self._std_weight_position * measurement[3],
            1e-2,
            2 * self._std_weight_position * measurement[3],
            10 * self._std_weight_velocity * measurement[3],
            10 * self._std_weight_velocity * measurement[3],
            1e-5,
            10 * self._std_weight_velocity * measurement[3],
        ]
        covariance = np.diag(np.square(std))
        return mean, covariance

    def predict(self, mean, covariance):
        std_pos = [
            self._std_weight_position * mean[3],
            self._std_weight_position * mean[3],
            1e-2,
            self._std_weight_position * mean[3],
        ]
        std_vel = [
            self._std_weight_velocity * mean[3],
            self._std_weight_velocity * mean[3],
            1e-5,
            self._std_weight_velocity * mean[3],
        ]
        motion_cov = np.diag(np.square(np.r_[std_pos, std_vel]))
        mean = np.dot(self._motion_mat, mean)
        covariance = np.linalg.multi_dot((self._motion_mat, covariance, self._motion_mat.T)) + motion_cov
        return mean, covariance

    def project(self, mean, covariance):
        std = [
            self._std_weight_position * mean[3],
            self._std_weight_position * mean[3],
            1e-1,
            self._std_weight_position * mean[3],
        ]
        innovation_cov = np.diag(np.square(std))
        mean = np.dot(self._update_mat, mean)
        covariance = np.linalg.multi_dot((self._update_mat, covariance, self._update_mat.T))
        return mean, covariance + innovation_cov

    def update(self, mean, covariance, measurement):
        projected_mean, projected_cov = self.project(mean, covariance)
        chol_factor, lower = scipy.linalg.cho_factor(projected_cov, lower=True, check_finite=False)
        kalman_gain = scipy.linalg.cho_solve(
            (chol_factor, lower), np.dot(covariance, self._update_mat.T).T, check_finite=False
        ).T
        innovation = measurement - projected_mean
        new_mean = mean + np.dot(innovation, kalman_gain.T)
        new_covariance = covariance - np.linalg.multi_dot((kalman_gain, projected_cov, kalman_gain.T))
        return new_mean, new_covariance

    def gating_distance(self, mean, covariance, measurements, only_position=False):
        mean, covariance = self.project(mean, covariance)
        if only_position:
            mean, covariance = mean[:2], covariance[:2, :2]
            measurements = measurements[:, :2]
        cholesky_factor = np.linalg.cholesky(covariance)
        d = measurements - mean
        z = scipy.linalg.solve_triangular(cholesky_factor, d.T, lower=True, check_finite=False, overwrite_b=True)
        squared_maha = np.sum(z * z, axis=0)
        return squared_maha

In [8]:
INFTY_COST = 1e5


def min_cost_matching(distance_metric, max_distance, tracks, detections, track_indices=None, detection_indices=None):
    if track_indices is None:
        track_indices = np.arange(len(tracks))
    if detection_indices is None:
        detection_indices = np.arange(len(detections))

    if len(detection_indices) == 0 or len(track_indices) == 0:
        return [], track_indices, detection_indices

    cost_matrix = distance_metric(tracks, detections, track_indices, detection_indices)
    cost_matrix[cost_matrix > max_distance] = max_distance + 1e-5
    indices = linear_sum_assignment(cost_matrix)
    indices = np.asarray(indices)
    indices = np.transpose(indices)

    matches, unmatched_tracks, unmatched_detections = [], [], []
    for col, detection_idx in enumerate(detection_indices):
        if col not in indices[:, 1]:
            unmatched_detections.append(detection_idx)
    for row, track_idx in enumerate(track_indices):
        if row not in indices[:, 0]:
            unmatched_tracks.append(track_idx)
    for row, col in indices:
        track_idx = track_indices[row]
        detection_idx = detection_indices[col]
        if cost_matrix[row, col] > max_distance:
            unmatched_tracks.append(track_idx)
            unmatched_detections.append(detection_idx)
        else:
            matches.append((track_idx, detection_idx))
    return matches, unmatched_tracks, unmatched_detections


def matching_cascade(
    distance_metric, max_distance, cascade_depth, tracks, detections, track_indices=None, detection_indices=None
):
    if track_indices is None:
        track_indices = list(range(len(tracks)))
    if detection_indices is None:
        detection_indices = list(range(len(detections)))

    unmatched_detections = detection_indices
    matches = []
    for level in range(cascade_depth):
        if len(unmatched_detections) == 0:
            break
        track_indices_l = [k for k in track_indices if tracks[k].time_since_update == 1 + level]
        if len(track_indices_l) == 0:
            continue
        matches_l, _, unmatched_detections = min_cost_matching(
            distance_metric, max_distance, tracks, detections, track_indices_l, unmatched_detections
        )
        matches += matches_l
    unmatched_tracks = list(set(track_indices) - set(k for k, _ in matches))
    return matches, unmatched_tracks, unmatched_detections


def gate_cost_matrix(
    kf, cost_matrix, tracks, detections, track_indices, detection_indices, gated_cost=INFTY_COST, only_position=False
):
    gating_dim = 2 if only_position else 4
    gating_threshold = chi2inv95[gating_dim]
    measurements = np.asarray([detections[i].to_xyah() for i in detection_indices])
    for row, track_idx in enumerate(track_indices):
        track = tracks[track_idx]
        gating_distance = kf.gating_distance(track.mean, track.covariance, measurements, only_position)
        cost_matrix[row, gating_distance > gating_threshold] = gated_cost
    return cost_matrix


def iou(bbox, candidates):
    """Computer intersection over union."""
    bbox_tl, bbox_br = bbox[:2], bbox[:2] + bbox[2:]
    candidates_tl = candidates[:, :2]
    candidates_br = candidates[:, :2] + candidates[:, 2:]

    tl = np.c_[
        np.maximum(bbox_tl[0], candidates_tl[:, 0])[:, np.newaxis],
        np.maximum(bbox_tl[1], candidates_tl[:, 1])[:, np.newaxis],
    ]
    br = np.c_[
        np.minimum(bbox_br[0], candidates_br[:, 0])[:, np.newaxis],
        np.minimum(bbox_br[1], candidates_br[:, 1])[:, np.newaxis],
    ]
    wh = np.maximum(0.0, br - tl)

    area_intersection = wh.prod(axis=1)
    area_bbox = bbox[2:].prod()
    area_candidates = candidates[:, 2:].prod(axis=1)
    return area_intersection / (area_bbox + area_candidates - area_intersection)


def iou_cost(tracks, detections, track_indices=None, detection_indices=None):
    if track_indices is None:
        track_indices = np.arange(len(tracks))
    if detection_indices is None:
        detection_indices = np.arange(len(detections))

    cost_matrix = np.zeros((len(track_indices), len(detection_indices)))
    for row, track_idx in enumerate(track_indices):
        if tracks[track_idx].time_since_update > 1:
            cost_matrix[row, :] = INFTY_COST
            continue
        bbox = tracks[track_idx].to_tlwh()
        candidates = np.asarray([detections[i].tlwh for i in detection_indices])
        cost_matrix[row, :] = 1.0 - iou(bbox, candidates)
    return cost_matrix

In [9]:
def _pdist(a, b):
    a, b = np.asarray(a), np.asarray(b)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    a2, b2 = np.square(a).sum(axis=1), np.square(b).sum(axis=1)
    r2 = -2.0 * np.dot(a, b.T) + a2[:, None] + b2[None, :]
    r2 = np.clip(r2, 0.0, float(np.inf))
    return r2


def _cosine_distance(a, b, data_is_normalized=False):
    if not data_is_normalized:
        a = np.asarray(a) / np.linalg.norm(a, axis=1, keepdims=True)
        b = np.asarray(b) / np.linalg.norm(b, axis=1, keepdims=True)
    return 1.0 - np.dot(a, b.T)


def _nn_euclidean_distance(x, y):
    distances = _pdist(x, y)
    return np.maximum(0.0, distances.min(axis=0))


def _nn_cosine_distance(x, y):
    distances = _cosine_distance(x, y)
    return distances.min(axis=0)


class NearestNeighborDistanceMetric(object):
    def __init__(self, metric, matching_threshold, budget=None):
        if metric == "euclidean":
            self._metric = _nn_euclidean_distance
        elif metric == "cosine":
            self._metric = _nn_cosine_distance
        else:
            raise ValueError("Invalid metric; must be either 'euclidean' or 'cosine'")
        self.matching_threshold = matching_threshold
        self.budget = budget
        self.samples = {}

    def partial_fit(self, features, targets, active_targets):
        for feature, target in zip(features, targets):
            self.samples.setdefault(target, []).append(feature)
            if self.budget is not None:
                self.samples[target] = self.samples[target][-self.budget :]
        self.samples = {k: self.samples[k] for k in active_targets}

    def distance(self, features, targets):
        cost_matrix = np.zeros((len(targets), len(features)))
        for i, target in enumerate(targets):
            cost_matrix[i, :] = self._metric(self.samples[target], features)
        return cost_matrix


In [10]:
class TrackState:
    Tentative = 1
    Confirmed = 2
    Deleted = 3


class Track:
    def __init__(self, mean, covariance, track_id, n_init, max_age, feature=None, class_name=None):
        self.mean = mean
        self.covariance = covariance
        self.track_id = track_id
        self.hits = 1
        self.age = 1
        self.time_since_update = 0
        self.state = TrackState.Tentative
        self.features = []
        if feature is not None:
            self.features.append(feature)
        self._n_init = n_init
        self._max_age = max_age
        self.class_name = class_name

    def to_tlwh(self):
        ret = self.mean[:4].copy()
        ret[2] *= ret[3]
        ret[:2] -= ret[2:] / 2
        return ret

    def to_tlbr(self):
        ret = self.to_tlwh()
        ret[2:] = ret[:2] + ret[2:]
        return ret

    def get_class(self):
        return self.class_name

    def predict(self, kf):
        self.mean, self.covariance = kf.predict(self.mean, self.covariance)
        self.age += 1
        self.time_since_update += 1

    def update(self, kf, detection):
        self.mean, self.covariance = kf.update(self.mean, self.covariance, detection.to_xyah())
        self.features.append(detection.feature)
        self.hits += 1
        self.time_since_update = 0
        if self.state == TrackState.Tentative and self.hits >= self._n_init:
            self.state = TrackState.Confirmed

    def mark_missed(self):
        if self.state == TrackState.Tentative:
            self.state = TrackState.Deleted
        elif self.time_since_update > self._max_age:
            self.state = TrackState.Deleted

    def is_tentative(self):
        return self.state == TrackState.Tentative

    def is_confirmed(self):
        return self.state == TrackState.Confirmed

    def is_deleted(self):
        return self.state == TrackState.Deleted


In [11]:
class Tracker:
    def __init__(self, metric, max_iou_distance=0.7, max_age=30, n_init=3):
        self.metric = metric
        self.max_iou_distance = max_iou_distance
        self.max_age = max_age
        self.n_init = n_init
        self.kf = KalmanFilter()
        self.tracks = []
        self._next_id = 1

    def predict(self):
        for track in self.tracks:
            track.predict(self.kf)

    def update(self, detections):
        matches, unmatched_tracks, unmatched_detections = self._match(detections)
        for track_idx, detection_idx in matches:
            self.tracks[track_idx].update(self.kf, detections[detection_idx])
        for track_idx in unmatched_tracks:
            self.tracks[track_idx].mark_missed()
        for detection_idx in unmatched_detections:
            self._initiate_track(detections[detection_idx])
        self.tracks = [t for t in self.tracks if not t.is_deleted()]

        active_targets = [t.track_id for t in self.tracks if t.is_confirmed()]
        features, targets = [], []
        for track in self.tracks:
            if not track.is_confirmed():
                continue
            features += track.features
            targets += [track.track_id for _ in track.features]
            track.features = []
        self.metric.partial_fit(np.asarray(features), np.asarray(targets), active_targets)

    def _match(self, detections):
        def gated_metric(tracks, dets, track_indices, detection_indices):
            features = np.array([dets[i].feature for i in detection_indices])
            targets = np.array([tracks[i].track_id for i in track_indices])
            cost_matrix = self.metric.distance(features, targets)
            cost_matrix = gate_cost_matrix(self.kf, cost_matrix, tracks, dets, track_indices, detection_indices)
            return cost_matrix

        confirmed_tracks = [i for i, t in enumerate(self.tracks) if t.is_confirmed()]
        unconfirmed_tracks = [i for i, t in enumerate(self.tracks) if not t.is_confirmed()]

        matches_a, unmatched_tracks_a, unmatched_detections = matching_cascade(
            gated_metric, self.metric.matching_threshold, self.max_age, self.tracks, detections, confirmed_tracks
        )

        iou_track_candidates = unconfirmed_tracks + [
            k for k in unmatched_tracks_a if self.tracks[k].time_since_update == 1
        ]
        unmatched_tracks_a = [k for k in unmatched_tracks_a if self.tracks[k].time_since_update != 1]
        matches_b, unmatched_tracks_b, unmatched_detections = min_cost_matching(
            iou_cost, self.max_iou_distance, self.tracks, detections, iou_track_candidates, unmatched_detections
        )

        matches = matches_a + matches_b
        unmatched_tracks = list(set(unmatched_tracks_a + unmatched_tracks_b))
        return matches, unmatched_tracks, unmatched_detections

    def _initiate_track(self, detection):
        mean, covariance = self.kf.initiate(detection.to_xyah())
        class_name = detection.get_class()
        self.tracks.append(
            Track(mean, covariance, self._next_id, self.n_init, self.max_age, detection.feature, class_name)
        )
        self._next_id += 1

In [12]:
def draw_bbox(
    image,
    bboxes,
    CLASSES,
    show_label=True,
    show_confidence=True,
    Text_colors=(255, 255, 0),
    rectangle_colors="",
    tracking=False,
):
    NUM_CLASS = CLASSES
    num_classes = len(NUM_CLASS)
    image_h, image_w, _ = image.shape
    hsv_tuples = [(1.0 * x / num_classes, 1.0, 1.0) for x in range(num_classes)]

    colors = list(map(lambda x: colorsys.hsv_to_rgb(*x), hsv_tuples))
    colors = list(map(lambda x: (int(x[0] * 255), int(x[1] * 255), int(x[2] * 255)), colors))

    random.seed(0)
    random.shuffle(colors)
    random.seed(None)

    for i, bbox in enumerate(bboxes):
        coor = np.array(bbox[:4], dtype=np.int32)
        score = bbox[4]
        class_ind = int(bbox[5])

        bbox_color = rectangle_colors if rectangle_colors != "" else colors[class_ind]
        bbox_thick = int(0.6 * (image_h + image_w) / 1000)

        if bbox_thick < 1:
            bbox_thick = 1
        fontScale = 0.75 * bbox_thick
        (x1, y1), (x2, y2) = (coor[0], coor[1]), (coor[2], coor[3])

        cv2.rectangle(image, (x1, y1), (x2, y2), bbox_color, bbox_thick * 2)

        if show_label:
            score_str = " {:.2f}".format(score) if show_confidence else ""
            if tracking:
                score_str = " " + str(score)
            label = "{}".format(NUM_CLASS[class_ind]) + score_str

            (text_width, text_height), baseline = cv2.getTextSize(
                label, cv2.FONT_HERSHEY_COMPLEX_SMALL, fontScale, thickness=bbox_thick
            )
            cv2.rectangle(
                image, (x1, y1), (x1 + text_width, y1 - text_height - baseline), bbox_color, thickness=cv2.FILLED
            )
            cv2.putText(
                image,
                label,
                (x1, y1 - 4),
                cv2.FONT_HERSHEY_COMPLEX_SMALL,
                fontScale,
                Text_colors,
                bbox_thick,
                lineType=cv2.LINE_AA,
            )

    return image

In [13]:
def yolov11_detections_to_deepsort_format(yolo_results, encoder, original_frame):
    """
    Convert YOLOv11 detection results to DeepSort format

    Args:
        yolo_results: Results from YOLO model prediction
        encoder: DeepSort feature encoder
        original_frame: Original frame (BGR format)

    Returns:
        List of Detection objects ready for DeepSort tracker
    """
    boxes = []
    names = []
    scores = []

    # Extract detections from YOLOv11 results
    for result in yolo_results:
        for box in result.boxes:
            # Get bounding box coordinates (xyxy format from YOLOv11)
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

            # Convert to tlwh format (top-left x, top-left y, width, height)
            width = x2 - x1
            height = y2 - y1

            # Get confidence and class
            confidence = float(box.conf[0])
            class_id = int(box.cls[0])
            class_name = result.names[class_id]

            # Append to lists
            boxes.append([x1, y1, width, height])
            names.append(class_name)
            scores.append(confidence)

    # Convert to numpy arrays
    boxes = np.array(boxes) if len(boxes) > 0 else np.array([])
    names = np.array(names) if len(names) > 0 else np.array([])
    scores = np.array(scores) if len(scores) > 0 else np.array([])

    # Extract features using DeepSort encoder
    if len(boxes) > 0:
        features = np.array(encoder(original_frame, boxes))
    else:
        features = np.array([])

    # Create Detection objects
    detections = [
        Detection(bbox, score, class_name, feature)
        for bbox, score, class_name, feature in zip(boxes, scores, names, features)
    ]

    return detections

In [14]:
def object_tracking_yolov11(
    video_path,
    output_path,
    yolo_model_path="yolo11n.pt",
    encoder_model_path=None,
    confidence_threshold=0.5,
):
    """
    Object tracking using YOLOv11 for detection and DeepSort for tracking

    Args:
        video_path: Path to input video
        output_path: Path to output video
        yolo_model_path: Path to YOLOv11 model weights
        encoder_model_path: Path to DeepSort feature encoder model
        confidence_threshold: Minimum confidence threshold for detections
    """

    # COCO class names (YOLOv11 uses COCO dataset)
    NUM_CLASS = {
        0: "person",
        1: "bicycle",
        2: "car",
        3: "motorcycle",
        4: "airplane",
        5: "bus",
        6: "train",
        7: "truck",
        8: "boat",
        9: "traffic light",
        10: "fire hydrant",
        11: "stop sign",
        12: "parking meter",
        13: "bench",
        14: "bird",
        15: "cat",
        16: "dog",
        17: "horse",
        18: "sheep",
        19: "cow",
        20: "elephant",
        21: "bear",
        22: "zebra",
        23: "giraffe",
        24: "backpack",
        25: "umbrella",
        26: "handbag",
        27: "tie",
        28: "suitcase",
        29: "frisbee",
        30: "skis",
        31: "snowboard",
        32: "sports ball",
        33: "kite",
        34: "baseball bat",
        35: "baseball glove",
        36: "skateboard",
        37: "surfboard",
        38: "tennis racket",
        39: "bottle",
        40: "wine glass",
        41: "cup",
        42: "fork",
        43: "knife",
        44: "spoon",
        45: "bowl",
        46: "banana",
        47: "apple",
        48: "sandwich",
        49: "orange",
        50: "broccoli",
        51: "carrot",
        52: "hot dog",
        53: "pizza",
        54: "donut",
        55: "cake",
        56: "chair",
        57: "couch",
        58: "potted plant",
        59: "bed",
        60: "dining table",
        61: "toilet",
        62: "tv",
        63: "laptop",
        64: "mouse",
        65: "remote",
        66: "keyboard",
        67: "cell phone",
        68: "microwave",
        69: "oven",
        70: "toaster",
        71: "sink",
        72: "refrigerator",
        73: "book",
        74: "clock",
        75: "vase",
        76: "scissors",
        77: "teddy bear",
        78: "hair drier",
        79: "toothbrush",
    }

    # Initialize YOLOv11 model
    print("Loading YOLOv11 model...")
    yolo_model = YOLO(yolo_model_path)

    # Initialize DeepSort
    print("Loading DeepSort encoder...")
    encoder = create_box_encoder(batch_size=32)  # Not using mars-small128.pb anymore, it was causing my kernel to crash

    # DeepSort parameters
    max_cosine_distance = 0.7
    nn_budget = None
    metric = NearestNeighborDistanceMetric("cosine", max_cosine_distance, nn_budget)
    tracker = Tracker(metric, max_iou_distance=0.7, max_age=30, n_init=3)

    # Open video
    print(f"Opening video: {video_path}")
    vid = cv2.VideoCapture(video_path)

    if not vid.isOpened():
        print(f"Error: Could not open video {video_path}")
        return

    # Get video properties
    width = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(vid.get(cv2.CAP_PROP_FPS))
    total_frames = int(vid.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"Video info: {width}x{height} @ {fps}fps, {total_frames} frames")

    # Create video writer
    codec = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, codec, fps, (width, height))

    times, times_2 = [], []
    frame_count = 0

    print("Processing video...")

    while True:
        ret, frame = vid.read()
        if not ret:
            break

        frame_count += 1
        original_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # YOLOv11 Detection
        t1 = time.time()

        # Run YOLOv11 detection with confidence threshold
        yolo_results = yolo_model(original_frame, conf=confidence_threshold, verbose=False)

        t2 = time.time()

        # Convert YOLOv11 detections to DeepSort format
        detections = yolov11_detections_to_deepsort_format(yolo_results, encoder, original_frame)

        # Update tracker
        tracker.predict()
        tracker.update(detections)

        # Prepare bounding boxes for visualization
        tracked_bboxes = []
        for track in tracker.tracks:
            if not track.is_confirmed() or track.time_since_update > 5:
                continue

            bbox = track.to_tlbr()  # Get bounding box
            class_name = track.get_class()  # Get class name
            tracking_id = track.track_id  # Get tracking ID

            # Find class index
            class_index = None
            for idx, name in NUM_CLASS.items():
                if name == class_name:
                    class_index = idx
                    break

            if class_index is not None:
                tracked_bboxes.append(bbox.tolist() + [tracking_id, class_index])

        # Draw bounding boxes with tracking IDs
        image = draw_bbox(original_frame, tracked_bboxes, CLASSES=NUM_CLASS, tracking=True)

        t3 = time.time()

        # Calculate FPS
        times.append(t2 - t1)
        times_2.append(t3 - t1)
        times = times[-20:]
        times_2 = times_2[-20:]

        ms = sum(times) / len(times) * 1000
        fps_detection = 1000 / ms
        fps_total = 1000 / (sum(times_2) / len(times_2) * 1000)

        # Add FPS text to frame
        image = cv2.putText(
            image,
            f"Detection FPS: {fps_detection:.1f}, Total FPS: {fps_total:.1f}",
            (10, 30),
            cv2.FONT_HERSHEY_COMPLEX_SMALL,
            1,
            (0, 255, 0),
            2,
        )

        # Convert back to BGR for video writing
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # Write frame
        out.write(image)

        # Print progress
        if frame_count % 30 == 0:
            print(
                f"Processed {frame_count}/{total_frames} frames "
                f"({100 * frame_count / total_frames:.1f}%) - "
                f"Detection: {fps_detection:.1f} FPS, Total: {fps_total:.1f} FPS"
            )

    # Cleanup
    out.release()
    vid.release()
    print(f"\nTracking complete! Output saved to: {output_path}")

Load the YOLOv11 model and run tracking

In [15]:
video_path = "soccer.mp4"
output_path = "output_tracked.mp4"

# from: https://github.com/nwojke/deep_sort/blob/master/mars-small128.pb
# encoder_model_path = "mars-small128.pb" # Was causing crashes...

object_tracking_yolov11(video_path="soccer.mp4", output_path="output_tracked.mp4")

Loading YOLOv11 model...
Loading DeepSort encoder...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /home/vscode/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 53.5MB/s]


Opening video: soccer.mp4
Video info: 1280x720 @ 25fps, 1074 frames
Processing video...
Processed 30/1074 frames (2.8%) - Detection: 79.0 FPS, Total: 17.6 FPS
Processed 60/1074 frames (5.6%) - Detection: 77.1 FPS, Total: 14.9 FPS
Processed 90/1074 frames (8.4%) - Detection: 75.5 FPS, Total: 21.1 FPS
Processed 120/1074 frames (11.2%) - Detection: 71.6 FPS, Total: 20.6 FPS
Processed 150/1074 frames (14.0%) - Detection: 56.6 FPS, Total: 10.2 FPS
Processed 180/1074 frames (16.8%) - Detection: 56.5 FPS, Total: 4.7 FPS
Processed 210/1074 frames (19.6%) - Detection: 73.5 FPS, Total: 5.9 FPS
Processed 240/1074 frames (22.3%) - Detection: 44.8 FPS, Total: 6.0 FPS
Processed 270/1074 frames (25.1%) - Detection: 43.9 FPS, Total: 5.3 FPS
Processed 300/1074 frames (27.9%) - Detection: 43.5 FPS, Total: 6.6 FPS
Processed 330/1074 frames (30.7%) - Detection: 42.5 FPS, Total: 6.7 FPS
Processed 360/1074 frames (33.5%) - Detection: 34.1 FPS, Total: 8.6 FPS
Processed 390/1074 frames (36.3%) - Detection: 49

KeyboardInterrupt: 